In [1]:
import pandas as pd
import numpy as np

## Load Station2_labelled.csv

In [2]:
df = pd.read_csv('../data/processed/station2_labelled.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25429 entries, 0 to 25428
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   datetime       25429 non-null  datetime64[us]
 1   DO             25429 non-null  float64       
 2   PH             25429 non-null  float64       
 3   AMMONIA(mg/l)  25429 non-null  float64       
 4   TEMP           25429 non-null  float64       
 5   NITRATE(PPM)   25429 non-null  float64       
 6   TURBIDITY      25429 non-null  float64       
 7   feed_label     25429 non-null  int64         
dtypes: datetime64[us](1), float64(6), int64(1)
memory usage: 1.6 MB


## Stress Indices

##### 1. DO_stess
Hossain et al. (2020) Machine Learning in Aquaculture in Reviews in Aquaculture engineer dissolved oxygen deficit.<br>
Ewaid et al. (2020) Water Quality Classification Based on Physicochemical Parameters Using Machine Learning in Water journal

In [4]:
df['DO_stress'] = np.maximum(0, 5.0 - df['DO'])

##### 2. pH Deviation (pH_dev)
Ewaid et al. (2020) Water Quality Classification Based on Physicochemical Parameters Using Machine Learning in Water journal

In [5]:
df['pH_dev'] = np.abs(df['PH'] - 7.5)

##### 3. Thermal Deviation (thermal_stress)
Brown et al. 1972

In [6]:
df['thermal_stress'] = (np.maximum(0, 22.0 - df['TEMP']) +
                        np.maximum(0, df['TEMP'] - 30.0))

##### 4. Chemical Transformation (NH3_toxic)
Hargreaves & Tucker (2004): toxic NH3 fraction via Henderson-Hasselbalchi<br>
pKa of ammonia at 25°C = 9.25 - same ammonia reading is far more toxic at high pH<br>
Johansson et al. 2018; Henderson-Hasselbalch

In [7]:
df['NH3_toxic'] = df['AMMONIA(mg/l)'] / (1 + 10 ** (9.25 - df['PH']))

##### 5. Composite Metabolic Stress Score (CMSS)

Tyagi et al. (2020) in WQI: Predicting and Evaluating Water Quality Using ML use a Weighted Arithmetic WQI - mathematically identical to CMSS — as a feature alongside individual parameter sub-indices. They report it captures compound stress events that individual features miss.

In [8]:
# speed-of-effect hierarchy: DO > NH3 > pH > thermal
df['CMSS'] = (0.35 * df['DO_stress']      +
              0.25 * df['NH3_toxic'] * 10  +
              0.15 * df['pH_dev']          +
              0.15 * df['thermal_stress'])

##### 6. Temporal
Zhu et al. (2019): DO follows diurnal cycle - same reading means
different things at 5AM vs 2PM

In [10]:
# cyclical encoding
df['hour'] = df['datetime'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

In [11]:
df[['DO_stress', 'pH_dev', 'thermal_stress', 'NH3_toxic','CMSS','hour_sin', 'hour_cos']].head()

,DO_stress,pH_dev,thermal_stress,NH3_toxic,CMSS,hour_sin,hour_cos
0,0.0,1.844,0.0,0.000006,0.276614,0.000000,1.000000
1,0.0,1.642,0.0,0.000029,0.246372,0.000000,1.000000
2,0.0,2.046,0.0,0.000009,0.306923,0.000000,1.000000
3,0.0,2.147,0.0,0.000002,0.322056,0.258819,0.965926
4,0.0,2.450,0.0,0.000004,0.367510,0.258819,0.965926


In [15]:
feature_engineered = df[[
    'NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY','NH3_toxic', 
    'DO_stress', 'pH_dev', 'thermal_stress', 'CMSS', 'hour_sin', 'hour_cos'
]]

In [16]:
feature_engineered.info()

<class 'pandas.DataFrame'>
RangeIndex: 25429 entries, 0 to 25428
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NITRATE(PPM)    25429 non-null  float64
 1   PH              25429 non-null  float64
 2   AMMONIA(mg/l)   25429 non-null  float64
 3   TEMP            25429 non-null  float64
 4   DO              25429 non-null  float64
 5   TURBIDITY       25429 non-null  float64
 6   NH3_toxic       25429 non-null  float64
 7   DO_stress       25429 non-null  float64
 8   pH_dev          25429 non-null  float64
 9   thermal_stress  25429 non-null  float64
 10  CMSS            25429 non-null  float64
 11  hour_sin        25429 non-null  float64
 12  hour_cos        25429 non-null  float64
dtypes: float64(13)
memory usage: 2.5 MB


## Save CSV

In [17]:
df.to_csv("../data/engineered/feature_engineered.csv", index=False)